In this notebook we aim to develop a protocole to evaluate OLAF relation extractionn  compoenents:


To achieve this task , we will follow this steps:

- Select a corpus.
- Select and create relevent concepts from the corpus.
- Create several pipelines with different components and parameters.
- Run all the pipelines.
- Find concepts involved in complete triples (relation with no null source and destination concepts) for each pipeline.
- Etablish the matching percentage of found concepts compared to selected concepts on step 2.


In [30]:
import spacy
from typing import Set, List
import pandas as pd
from olaf import Pipeline
from olaf.commons.logging_config import logger
from olaf.data_container import CandidateTerm, Relation, Concept
from olaf.data_container.knowledge_representation_schema import KnowledgeRepresentation
from olaf.pipeline.pipeline_component.term_extraction import (
    POSTermExtraction,
    TFIDFTermExtraction,
    ManualCandidateTermExtraction
    )
from olaf.pipeline.pipeline_component.concept_relation_extraction import (
    CTsToConceptExtraction,
    CTsToRelationExtraction,
    SynonymRelationExtraction,
    SynonymConceptExtraction,
    AgglomerativeClusteringRelationExtraction,
    AgglomerativeClusteringConceptExtraction,
    LLMBasedRelationExtraction
)
from olaf.commons.spacy_processing_tools import is_not_punct, is_not_stopword, select_on_pos

from olaf.pipeline.pipeline_component.candidate_term_enrichment import SemanticBasedEnrichment

from olaf.repository.corpus_loader.text_corpus_loader import TextCorpusLoader

In [31]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from matplotlib_venn import venn2, venn3

In [32]:
nlp = spacy.load("en_core_web_lg")

In [33]:
import torch, gc
def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

# Select Corpus

In [34]:
corpus_path = "GC10-DET_doc.txt"
corpus_loader = TextCorpusLoader(corpus_path)

# Select manually and create relevent relation from the corpus.


In [35]:
import json
import re

def format_concept_or_label(text):
    # Remplacer les underscores par des espaces et transformer en minuscules
    return text.replace('_', ' ').lower()

def format_camel_case(text):
    # Remplacer les underscores par des espaces et transformer en minuscules
    text = text.replace('_', ' ')
    # Ajouter des espaces entre les mots en camel case
    text = re.sub(r'(?<!^)(?=[A-Z])', ' ', text).lower()
    return text

expected_concepts = []
with open("concepts.txt", 'r') as f:
    lines = f.readlines()
    expected_concepts = [concept.rstrip("\n") for concept in lines]
    expected_concepts = [Concept(concept) for concept in expected_concepts]
    f.close()


with open("relations.json", 'r', encoding='utf-8') as file:
    expected_relations = json.load(file)

expected_relations = [(format_concept_or_label(concept_source),
                       format_camel_case(relation_label),
                       format_concept_or_label(concept_destination))
                      for concept_source, relation_label, concept_destination in expected_relations]


expected_relations = { Relation(relation[1], Concept(relation[0]), Concept(relation[2])) for relation in expected_relations}
expected_relations

{(product, has abnormal, appearance),
 (steel strip, has abnormal, appearance),
 (crease, has appearance, vertical),
 (waist folding, has appearance, wrinkles like),
 (silk spot, has appearance, wave like plaque),
 (defect, has appearance, appearance),
 (inclusion, has appearance, spot),
 (crescent gap, has appearance, half circle),
 (rolled pit, has appearance, periodic bulges or pits),
 (defect, is caused by, cause),
 (punching, is caused by, mechanical failure),
 (crescent gap, is caused by, cutting),
 (crease, is caused by, local yield),
 (waist folding, is caused by, low carbon),
 (oil spot, is caused by, mechanical lubricant),
 (water spot, is caused by, drying),
 (factory, is part of, factory),
 (roller, is part of, machine),
 (machine, is part of, factory),
 (production line, is part of, factory),
 (product, is produced by, factory),
 (steel strip, is produced by, machine)}

In [36]:
labels = set([relation.label for relation in expected_relations])
labels

{'has abnormal',
 'has appearance',
 'is caused by',
 'is part of',
 'is produced by'}

# Testing relation ratio function

In [190]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util




sentence_transformer_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
comparator_args={"threshold": 0.6}

def hg_lm_similaritiry(embedding_a : str, embedding_b: str, threshold=.8):
    return util.pytorch_cos_sim(embedding_a, embedding_b) > threshold

def create_concepts_embedings(concepts: List[Concept], model=sentence_transformer_model) -> List[np.ndarray]:
    concept_labels = [concept.label for concept in concepts]
    concept_embedings = model.encode(concept_labels)
    return concept_embedings


def create_relations_embedings(relations: List[Relation], model=sentence_transformer_model) -> List[np.ndarray]:
    return [
            (
                model.encode(relation.source_concept.label), 
                # model.encode(relation.label), 
                model.encode(relation.destination_concept.label)
            ) for relation in relations
            ]


def get_unexpected_concepts(concepts: List[Concept], expected_concepts : List[Concept]):
    concepts = list(concepts)
    concepts_embedings = create_concepts_embedings(concepts)
    expected_concepts_embeding = create_concepts_embedings(expected_concepts)
    return [
        concepts[idc]
        for idc, concept_embeding in enumerate(concepts_embedings)
        if all(
            hg_lm_similaritiry(concept_embeding, expected_concept_embeding)
            <= 0.7
            for expected_concept_embeding in expected_concepts_embeding
        )
    ]

def is_valid_relation(relation : Relation):
    return relation.source_concept is not None and relation.destination_concept is not None


def are_equivalent(rel_embeding_a : np.ndarray, rel_embeding_b : np.ndarray) -> bool:
    return all(hg_lm_similaritiry(embeding_a, embeding_b) for embeding_a, embeding_b in zip(rel_embeding_a, rel_embeding_b)) \
        or all(hg_lm_similaritiry(embeding_a, embeding_b) for embeding_a, embeding_b in zip(rel_embeding_b, rel_embeding_a))


def get_relation_ratio(pipeline : Pipeline, expected_relations : List[Relation], comparator = hg_lm_similaritiry, comparator_args:dict={}, verbose=False) -> tuple:
    """
    Calculate the ratio of expected and unexpected relations in a given pipeline.

    Parameters
    ----------
    pipeline : Pipeline
        The pipeline object containing relations.
    expected_relations : List[Relation]
        A list of expected relations.kwargs

    Returns
    -------
    Tuple[float, float]: A tuple containing:
        The percentage of expected relations found in the pipeline.
        The percentage of unexpected relations in the pipeline.
    """
    
    
    found_relations = pipeline.kr.relations
    if len(found_relations) <= 0:
        return (0, 0, 0)
    found_relations = [relation for relation in found_relations if is_valid_relation(relation)]
    expected_relations = list(expected_relations)
    cooccurrence_count = 0

    found_relations_embedings =  create_relations_embedings(found_relations)
    expected_relations_embeding =  create_relations_embedings(expected_relations)

    for idx1, r1 in enumerate(expected_relations_embeding):
        if verbose:
            print(f"\n{str(expected_relations[idx1])} : ", end= "")
        for idx2, r2 in enumerate(found_relations_embedings):
            if are_equivalent(r1, r2) :
                if verbose:
                    print(f"{found_relations[idx2]}, ", end= "")
                cooccurrence_count += 1
    if cooccurrence_count == 0:
        return (0, 0, 0)
    recall = cooccurrence_count/len(expected_relations)
    precision = cooccurrence_count/len(found_relations)
    f1 = 2*(precision * recall)/(precision+recall)
    return (precision, recall, f1)

In [188]:
l1 = ["a", "b"]
l2 = [1, 3]
zipped_list = list(zip(l1, l2))
print(zipped_list)

[('a', 1), ('b', 3)]


In [186]:
r1 = Relation("has appearance", Concept("crescent gap"),  Concept("half circle"))
r2 = Relation("is caused by", Concept("crescent gap"),  Concept("half circle"))
r2 = Relation("is caused by",  Concept("half circle"), Concept("crescent gap"))

relation1, relation2 = create_relations_embedings([r1, r2])

[hg_lm_similaritiry(embeding_a, embeding_b) for embeding_a, embeding_b in zip([relation1[0], relation1[-1]], [relation2[0], relation2[-1]])]

[tensor([[False]]), tensor([[False]])]

In [39]:
from olaf.pipeline.pipeline_component.term_extraction.manual_candidate_terms import (
    ManualCandidateTermExtraction,
)
from olaf.pipeline.pipeline_component.concept_relation_extraction.candidate_terms_to_concepts import CTsToConceptExtraction

from olaf.pipeline.pipeline_component.concept_relation_extraction.candidate_terms_to_relations import CTsToRelationExtraction


# concept extraction component
concepts = [
    "defect type",
    "steel strip surface",
    "punching",
    "mechanical failure",
    "welding line",
    "coil",
    "weld line",
    "crescent gap",
    "cutting",
    "water spot",
    "drying",
    "oil spot",
    "mechanical lubricant",
    "silk spot",
    "plaque",
    "strip surface",
    "roller",
    "pressure",
    "inclusion",
    "metal surface",
    "spots",
    "fish scale shape",
    "block irregular distribution",
    "rolled pit",
    "bulges",
    "pits",
    "steel plate",
    "work roll",
    "tension roll",
    "damage",
    "crease",
    "fold",
    "uncoiling process",
    "waist folding",
    "deformation",
    "low-carbon"
]

"""
    You are an helpful assistant helping building an ontology of technical documentation of quality defects.
    Extract the most meaningful words describing defects ans their causes, appearance of products. 
    we will use this list of relations to extract relations between concepts : ['has abnormal', 'has appearance', 'is caused by', 'is part of', 'is produced by']
    Keep only words that could be relations and not concepts.
    Write them as a python list of string with double quotes.
    
    Text: 
"""

relations = [
    "needs to be punched",
    "may lead to",
    "resulting in",
    "fold",
    "moving",
    "produced by",
    "are different",
    "detected by mistake",
    "described in detail",
    "explaining",
    "appears",
    "showing",
    "accompanied by",
    "fall off",
    "pressed into",
    "changed",
    "weld",
    "needs to be detected",
    "tracked",
    "circumvented",
    "indicating",
    "caused by",
    "affect",
    "distributed",
    "cutting",
    "appear"
]


ct_concept_label = { concept : {concept} for concept in concepts}

manuel_concept_extraction = ManualCandidateTermExtraction(
    ct_label_strings_map=ct_concept_label
)

concept_extraction = CTsToConceptExtraction(
)


# Usefull function

In [40]:
def display_concept(kr: KnowledgeRepresentation) -> None:
    print("Concepts in KR:")
    for concept in kr.concepts:
        print(concept.label)


def display_relation(kr: KnowledgeRepresentation) -> None:
    print("Relations in KR:")
    for relation in kr.relations:
        if (
            relation.source_concept is not None
            or relation.destination_concept is not None
        ):
            print(
                (
                    relation.source_concept.label,
                    relation.label,
                    relation.destination_concept.label,
                )
            )

def describe_pipeline(pipeline: Pipeline) -> None:
    print(pipeline.__class__.__name__)
    for component in pipeline.pipeline_components:
        print(f"\t {component.__class__.__name__}")

In [41]:
# Fonction pour créer les diagrammes en barres
def create_bar_chart(index_name, data):
    fig, ax = plt.subplots(figsize=(9, 5))
    bar_width = 0.2
    opacity = 0.8

    # Configurer les positions des barres
    r1 = np.arange(len(data.columns.levels[0]))
    r2 = [x + bar_width for x in r1]
    r3 = [x + bar_width for x in r2]

    precision = data.loc[index_name].xs('Precision', level=1)
    rappel = data.loc[index_name].xs('Rappel', level=1)
    f1 = data.loc[index_name].xs('F1', level=1)

    rects1 = ax.bar(r1, precision, bar_width, alpha=opacity, color='b', label='Précision')
    rects2 = ax.bar(r2, rappel, bar_width, alpha=opacity, color='g', label='Rappel')
    rects3 = ax.bar(r3, f1, bar_width, alpha=opacity, color='r', label='F1')

    ax.set_xlabel('Composants')
    ax.set_ylabel('Scores')
    ax.set_title(f'Scores de Précision, Rappel et F1 pour {index_name}')
    ax.set_xticks([r + bar_width for r in range(len(data.columns.levels[0]))])
    ax.set_xticklabels(data.columns.levels[0])
    ax.legend()

    plt.show()

def create_bar_chart(index_name, pipelines_scores):
    data = pipelines_scores.reset_index().melt(id_vars='index', var_name=['Composant', 'Métrique'], value_name='Score')
    data.rename(columns={'index': 'Extraction'}, inplace=True)

    df = data[data['Extraction'] == index_name]
    fig = px.bar(df, x='Composant', y='Score', color='Métrique', barmode='group',
                 title=f'Scores de Précision, Rappel et F1 pour {index_name}')
    
    fig.update_layout(
        xaxis_title='Composants',
        yaxis_title='Scores'
    )
    fig.update_layout(width=1000, height=600)
    fig.show()



# Creating pipelines

In [42]:
relation_extraction_components = ["CandidatToRelation", "SynonymToRelation", "AgglomerativeClustering"]
term_extraction_components = ["LLM Term Extraction", "POStag Term Extraction", "TFIDF Term Extraction"]
results = pd.DataFrame(
    index=relation_extraction_components,
    columns=term_extraction_components
    )

multi_index = pd.MultiIndex.from_product([
   relation_extraction_components, 
    ["Precision", "Rappel", "F1"]
    ])
pipelines_scores = pd.DataFrame(index=term_extraction_components, columns=multi_index)

pipelines_scores

CandidatToRelation             SynonymToRelation  \
                                Precision Rappel   F1         Precision   
LLM Term Extraction                   NaN    NaN  NaN               NaN   
POStag Term Extraction                NaN    NaN  NaN               NaN   
TFIDF Term Extraction                 NaN    NaN  NaN               NaN   

                                   AgglomerativeClustering              
                       Rappel   F1               Precision Rappel   F1  
LLM Term Extraction       NaN  NaN                     NaN    NaN  NaN  
POStag Term Extraction    NaN  NaN                     NaN    NaN  NaN  
TFIDF Term Extraction     NaN  NaN                     NaN    NaN  NaN

## LLM Term  Extraction

In [155]:
llm_pipelines = [None, None, None]
llm_results = np.zeros(9)


### LLM Term  Extraction and Candidat To Relation Extraction

In [161]:
idx = 0

In [169]:
# concept extraction component
concepts = [
    "defect type",
    "steel strip surface",
    "punching",
    "mechanical failure",
    "welding line",
    "coil",
    "weld line",
    "crescent gap",
    "cutting",
    "water spot",
    "drying",
    "oil spot",
    "mechanical lubricant",
    "silk spot",
    "plaque",
    "strip surface",
    "roller",
    "pressure",
    "inclusion",
    "metal surface",
    "spots",
    "fish scale shape",
    "block irregular distribution",
    "rolled pit",
    "bulges",
    "pits",
    "steel plate",
    "work roll",
    "tension roll",
    "damage",
    "crease",
    "fold",
    "uncoiling process",
    "waist folding",
    "deformation",
    "low-carbon"
]

ct_concept_label = { concept : {concept} for concept in concepts}



llm_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
            ManualCandidateTermExtraction(
                ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            ManualCandidateTermExtraction(
                ct_label_strings_map={ relation : {relation} for relation in relations}
            ),
            CTsToRelationExtraction(
                concept_max_distance=5
            )
        ],
        corpus_loader=corpus_loader
    )


free_gpu()
current_pipeline = llm_pipelines[idx]
current_pipeline.run()


llm_results[3*idx: 3*idx + 3]= list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args, verbose=True)
    )

print(results)

[2024-07-12 00:19:43,432] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:19:43,433] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]



is caused by : 
is caused by : 
is produced by : 
is produced by : 
has appearance : 
has abnormal : 
is part of : 
has appearance : 
is caused by : 
is caused by : 
has appearance : 
is caused by : 
is caused by : caused by, 
is part of : 
is caused by : produced by, 
is part of : 
has appearance : 
has appearance : 
has abnormal : 
has appearance : 
is part of : 
has appearance : (0.4, 0.09090909090909091, 0.14814814814814814)


In [170]:
idx += 1

### LLM Term  Extraction and Synonym Relation Extraction

In [172]:


llm_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
           ManualCandidateTermExtraction(
                ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            ManualCandidateTermExtraction(
                ct_label_strings_map={ relation : {relation} for relation in relations}
            ),
           SynonymRelationExtraction(
               concept_max_distance=5
           )
        ],
        corpus_loader=corpus_loader
    )



free_gpu()
current_pipeline = llm_pipelines[idx]
current_pipeline.run()


llm_results[3*idx: 3*idx + 3]= list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:20:04,730] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:20:04,731] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]


(0.4, 0.09090909090909091, 0.14814814814814814)


In [173]:
idx += 1

### LLM Term  Extraction and Agglomerative Clustering Reltation Extraction

In [178]:
llm_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
            ManualCandidateTermExtraction(
                ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            ManualCandidateTermExtraction(
                ct_label_strings_map={ relation : {relation} for relation in relations}
            ),
            AgglomerativeClusteringRelationExtraction(
                concept_max_distance=8
            )
        ],
        corpus_loader=corpus_loader
    )

free_gpu()
current_pipeline = llm_pipelines[idx]
current_pipeline.run()


llm_results[3*idx: 3*idx + 3] = list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:22:31,249] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:22:31,251] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2024-07-12 00:22:31,252] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:22:31,253] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for nb_clusters option, default will be set to 2.]
[2024-07-12 00:22:31,254] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]


(0.2857142857142857, 0.09090909090909091, 0.13793103448275862)


In [52]:
# candidate_terms = {}

# cterm_index = {cterm.label: cterm for cterm in candidate_terms}
# ct_str_list = "\n".join(cterm_index.keys())
# prompt = create_relation_prompt(candidate_terms)
# llm_output = ""

# create_relations(llm_output, cterm_index)

### Score des pipelines

In [179]:
llm_results

array([0.4       , 0.09090909, 0.14814815, 0.4       , 0.09090909,
       0.14814815, 0.28571429, 0.09090909, 0.13793103])

In [180]:
pipelines_scores.loc[term_extraction_components[0]] = llm_results
pipelines_scores

CandidatToRelation                      \
                                Precision    Rappel        F1   
LLM Term Extraction                   0.4  0.090909  0.148148   
POStag Term Extraction                0.0       0.0       0.0   
TFIDF Term Extraction            0.090909  0.136364  0.109091   

                       SynonymToRelation                      \
                               Precision    Rappel        F1   
LLM Term Extraction                  0.4  0.090909  0.148148   
POStag Term Extraction          0.090909  0.045455  0.060606   
TFIDF Term Extraction           0.242424  0.363636  0.290909   

                       AgglomerativeClustering                      
                                     Precision    Rappel        F1  
LLM Term Extraction                   0.285714  0.090909  0.137931  
POStag Term Extraction                     0.1  0.045455    0.0625  
TFIDF Term Extraction                 0.058824  0.045455  0.051282

In [181]:
create_bar_chart("LLM Term Extraction", pipelines_scores)

## POS tag Term Extraction

In [212]:
postag_pipelines = [None, None, None]
pos_results = np.ones(9)
idx = 0

### POS tag Term  extraction and Candidat To Concept Extraction

In [213]:

postag_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
            ManualCandidateTermExtraction(
            ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            POSTermExtraction(
                pos_selection=["VERB", "ADJ"]
            ),
            CTsToRelationExtraction(
                concept_max_distance=8
            )
        ],
        corpus_loader=corpus_loader
    )


free_gpu()
current_pipeline = postag_pipelines[idx]
current_pipeline.run()


pos_results[3*idx: 3*idx + 3]= list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:43:11,793] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:43:11,794] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2024-07-12 00:43:11,795] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2024-07-12 00:43:11,795] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]


(0.08333333333333333, 0.09090909090909091, 0.08695652173913043)


In [58]:
display_relation(current_pipeline.kr)

Relations in KR:
('crease', 'vertical', 'waist folding')
('inclusion', 'typical', 'metal surface')
('mechanical failure', 'unwanted', 'punching')
('rolled pit', 'rolled', 'pits')
('pits', 'periodic', 'pits')
('pits', 'periodic', 'bulges')
('mechanical failure', 'lead', 'punching')
('crease', 'transverse', 'waist folding')
('rolled pit', 'rolled', 'bulges')
('spots', 'produced', 'drying')
('punching', 'resulting', 'punching')
('roller', 'uneven', 'pressure')
('metal surface', 'showing', 'spots')


In [214]:
idx += 1

### POS tag Term  extraction and Synonym Concept Extraction

In [215]:
postag_pipelines[idx] = Pipeline(
    spacy_model=nlp,
    pipeline_components=[
        ManualCandidateTermExtraction(
            ct_label_strings_map=ct_concept_label
        ),
        AgglomerativeClusteringConceptExtraction(
            distance_threshold=.4
        ),
        POSTermExtraction(
            pos_selection=["VERB", "ADJ"]
        ),
        SynonymRelationExtraction(
            concept_max_distance=8
        )
    ],
    corpus_loader=corpus_loader
)

free_gpu()
current_pipeline = postag_pipelines[idx]
current_pipeline.run()


pos_results[3*idx: 3*idx + 3]= list(
    result:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:43:22,386] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:43:22,387] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2024-07-12 00:43:22,388] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2024-07-12 00:43:22,389] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]


(0.08333333333333333, 0.09090909090909091, 0.08695652173913043)


In [216]:
idx += 1

### POS tag Term  extraction and Agglomerative clustering Extraction

In [217]:
postag_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
            ManualCandidateTermExtraction(
                ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            POSTermExtraction(
                pos_selection=["VERB", "ADJ"]
            ),
            AgglomerativeClusteringRelationExtraction(
               concept_max_distance=8
           )
        ],
        corpus_loader=corpus_loader
    )



free_gpu()
current_pipeline = postag_pipelines[idx]
current_pipeline.run()


pos_results[3*idx: 3*idx + 3]= list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:43:29,213] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:43:29,213] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2024-07-12 00:43:29,214] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2024-07-12 00:43:29,215] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2024-07-12 00:43:29,216] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:43:29,216] [WARNING] [agglomerative_clustering_relation_extract

(0.10526315789473684, 0.09090909090909091, 0.0975609756097561)


In [218]:
pos_results

array([0.08333333, 0.09090909, 0.08695652, 0.08333333, 0.09090909,
       0.08695652, 0.10526316, 0.09090909, 0.09756098])

### Score des pipelines

In [219]:
pipelines_scores.loc[term_extraction_components[1]] = pos_results
pipelines_scores

CandidatToRelation                      \
                                Precision    Rappel        F1   
LLM Term Extraction                   0.4  0.090909  0.148148   
POStag Term Extraction           0.083333  0.090909  0.086957   
TFIDF Term Extraction            0.090909  0.136364  0.109091   

                       SynonymToRelation                      \
                               Precision    Rappel        F1   
LLM Term Extraction                  0.4  0.090909  0.148148   
POStag Term Extraction          0.083333  0.090909  0.086957   
TFIDF Term Extraction           0.242424  0.363636  0.290909   

                       AgglomerativeClustering                      
                                     Precision    Rappel        F1  
LLM Term Extraction                   0.285714  0.090909  0.137931  
POStag Term Extraction                0.105263  0.090909  0.097561  
TFIDF Term Extraction                 0.058824  0.045455  0.051282

In [220]:
create_bar_chart("POStag Term Extraction", pipelines_scores)

## TFIDF  Term Extraction

In [221]:
tfidf_pipelines = [None, None, None]
tfidf_results = np.ones(9)
idx = 0

In [222]:
from spacy.matcher import Matcher

def relation_postprocessor(relations : Set[Relation], nlp=nlp) -> Set[Relation]:
    correct_relations = set()
    relation_patterns = [
        [{"POS": "AUX", "DEP": "ROOT"}],
        [{"POS": "AUX", "OP": "?"}, {"POS": "ADV", "OP": "?"},{"POS": "VERB"}, {"POS": "ADP", "OP": "?"}],
        [{"POS": "AUX"}, {"POS": "ADJ", "OP": "+"}, {"POS": "ADP"}],
        [{"POS": "AUX"}, {"POS": "VERB", "OP": "+"}, {"POS": "ADP", "OP": "?"}],
        ]
    matcher = Matcher(nlp.vocab)

    matcher.add("REALTION_PATTERN", relation_patterns)

    for relation in relations:
        relation_doc = nlp(relation.label)
        matches = matcher(relation_doc)
        if any(
            len(relation_doc[start_idx:end_idx]) == len(relation_doc)
            for _, start_idx, end_idx in matches
        ):
            correct_relations.add(relation)
    
    return correct_relations


### TFIDF Term Extraction and Candidat To Concept Extraction

In [224]:

tfidf_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
           ManualCandidateTermExtraction(
                ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            TFIDFTermExtraction(max_term_token_length=4, cts_post_processing_functions=[relation_postprocessor]),
            CTsToRelationExtraction(
               concept_max_distance=8
           )
        ],
        corpus_loader=corpus_loader
    )


free_gpu()
current_pipeline = tfidf_pipelines[idx]
current_pipeline.run()


tfidf_results[3*idx: 3*idx + 3]= list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:45:09,926] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:45:09,927] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2024-07-12 00:45:09,928] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
/home/oumar/Bureau/ontology-learning/env/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:525: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'



(0.23529411764705882, 0.5454545454545454, 0.3287671232876712)


In [225]:
idx += 1

### TFIDF Term Extraction and Synonym Relation Extraction

In [227]:

tfidf_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
            ManualCandidateTermExtraction(
                ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            TFIDFTermExtraction(
                max_term_token_length=4,
                candidate_term_threshold=.01,
                cts_post_processing_functions=[relation_postprocessor]
            ),
            SynonymRelationExtraction(
                concept_max_distance=8
            )
        ],
        corpus_loader=corpus_loader
    )


free_gpu()
current_pipeline = tfidf_pipelines[idx]
current_pipeline.run()


tfidf_results[3*idx: 3*idx + 3]= list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:45:42,396] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:45:42,397] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2024-07-12 00:45:42,398] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
/home/oumar/Bureau/ontology-learning/env/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:525: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'



(0.23529411764705882, 0.5454545454545454, 0.3287671232876712)


In [228]:
idx += 1

### TFIDF Term Extraction and Agglomerative clustering Relation Extraction

In [233]:

tfidf_pipelines[idx] = Pipeline(
        spacy_model=nlp,
        pipeline_components=[
            ManualCandidateTermExtraction(
                ct_label_strings_map=ct_concept_label
            ),
            AgglomerativeClusteringConceptExtraction(
                distance_threshold=.4
            ),
            TFIDFTermExtraction(
                max_term_token_length=4, 
                candidate_term_threshold=.01,
                cts_post_processing_functions=[relation_postprocessor]
            ),
            AgglomerativeClusteringRelationExtraction(
                concept_max_distance=6
            )
        ],
        corpus_loader=corpus_loader
    )


free_gpu()
current_pipeline = tfidf_pipelines[idx]
current_pipeline.run()


tfidf_results[3*idx: 3*idx + 3]= list(
    results:=get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)
    )

print(results)

[2024-07-12 00:47:53,438] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:47:53,439] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2024-07-12 00:47:53,440] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2024-07-12 00:47:53,440] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:47:53,441] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for nb_clusters option, default will be set to 2.]
[2024-07-12 00:47:53,441] [WARNING] [agglomerat

(0.058823529411764705, 0.045454545454545456, 0.05128205128205128)


In [72]:
display_relation(current_pipeline.kr)

Relations in KR:
('pits', 'are', 'pits')
('pits', 'are', 'bulges')
('oil spot', 'caused by', 'mechanical lubricant')
('metal surface', 'usually showing', 'fish scale shape')
('work roll', 'roll', 'work roll')
('rolled pit', 'rolled', 'bulges')
('fish scale shape', 'shape', 'block irregular distribution')
('pits', 'surface', 'steel plate')
('rolled pit', 'are', 'pits')
('punching', 'resulting in', 'punching')
('bulges', 'surface', 'steel plate')
('inclusion', 'defect', 'metal surface')
('spots', 'produced', 'drying')
('crease', 'transverse', 'fold')
('work roll', 'roll', 'damage')
('mechanical failure', 'may', 'punching')
('metal surface', 'showing', 'spots')


In [73]:
idx

2

### Scores des pipelines

In [234]:
pipelines_scores.loc[term_extraction_components[2]] = tfidf_results
pipelines_scores

CandidatToRelation                      \
                                Precision    Rappel        F1   
LLM Term Extraction                   0.4  0.090909  0.148148   
POStag Term Extraction           0.083333  0.090909  0.086957   
TFIDF Term Extraction            0.235294  0.545455  0.328767   

                       SynonymToRelation                      \
                               Precision    Rappel        F1   
LLM Term Extraction                  0.4  0.090909  0.148148   
POStag Term Extraction          0.083333  0.090909  0.086957   
TFIDF Term Extraction           0.235294  0.545455  0.328767   

                       AgglomerativeClustering                      
                                     Precision    Rappel        F1  
LLM Term Extraction                   0.285714  0.090909  0.137931  
POStag Term Extraction                0.105263  0.090909  0.097561  
TFIDF Term Extraction                 0.058824  0.045455  0.051282

In [235]:
create_bar_chart("TFIDF Term Extraction", pipelines_scores)

In [243]:

# Assuming you have a dataframe named 'df'
for index, row in pipelines_scores.iterrows():
    # Access the values of each column in the row
    print(" & ".join(row.to_numpy()))
    # print()

TypeError: sequence item 0: expected str instance, float found

In [76]:

l = "0.121	0.9	0.213	0.121	0.9	0.213	0.125	1.0	0.222".split()
print(" & ".join(l))

0.121 & 0.9 & 0.213 & 0.121 & 0.9 & 0.213 & 0.125 & 1.0 & 0.222


# last attempt

In [236]:
llm_output = [
    ["water spot", "is produced by", "drying"],
    ["water spot", "is produced by", "production"],
    ["oil spot", "is caused by", "contamination"],
    ["contamination", "is caused by", "mechanical lubricant"],
    ["oil spot", "has appearance", "product"],
    ["crescent gap", "is caused by", "cutting"],
    ["weld line", "is part of", "strip"],
    ["inclusion", "has appearance", "small spots"],
    ["inclusion", "has appearance", "fish scale shape"],
    ["inclusion", "has appearance", "strip shape"],
    ["inclusion", "has appearance", "block irregular distribution"],
    ["inclusion", "is part of", "upper surface"],
    ["inclusion", "is part of", "lower surface"],
    ["inclusion", "is accompanied by", "rough pockmarked surfaces"],
    ["crease", "has appearance", "vertical transverse fold"],
    ["crease", "has abnormal", "spacing"],
    ["crease", "is caused by", "local yield"],
    ["crease", "is part of", "strip"],
    ["silk spot", "has appearance", "plaque"],
    ["silk spot", "is caused by", "uneven temperature"],
    ["silk spot", "is caused by", "uneven pressure"],
    ["waist folding", "has appearance", "obvious folds"],
    ["waist folding", "has appearance", "wrinkles"],
    ["waist folding", "is caused by", "local deformation"],
    ["waist folding", "is caused by", "low-carbon"],
    ["punching", "is produced by", "production line"],
    ["punching", "is produced by", "strip"],
    ["punching", "is caused by", "mechanical failure"],
    ["punctate", "is part of", "rolled pit"],
    ["rolled pit", "has appearance", "bulges"],
    ["rolled pit", "has appearance", "pits"],
    ["rolled pit", "is caused by", "work roll"],
    ["rolled pit", "is caused by", "tension roll"],
    ["rolled pit", "is part of", "steel plate"]
]


In [237]:
expected_relations

{(product, has abnormal, appearance),
 (steel strip, has abnormal, appearance),
 (crease, has appearance, vertical),
 (waist folding, has appearance, wrinkles like),
 (silk spot, has appearance, wave like plaque),
 (defect, has appearance, appearance),
 (inclusion, has appearance, spot),
 (crescent gap, has appearance, half circle),
 (rolled pit, has appearance, periodic bulges or pits),
 (defect, is caused by, cause),
 (punching, is caused by, mechanical failure),
 (crescent gap, is caused by, cutting),
 (crease, is caused by, local yield),
 (waist folding, is caused by, low carbon),
 (oil spot, is caused by, mechanical lubricant),
 (water spot, is caused by, drying),
 (factory, is part of, factory),
 (roller, is part of, machine),
 (machine, is part of, factory),
 (production line, is part of, factory),
 (product, is produced by, factory),
 (steel strip, is produced by, machine)}

In [238]:
expected_relations

{(product, has abnormal, appearance),
 (steel strip, has abnormal, appearance),
 (crease, has appearance, vertical),
 (waist folding, has appearance, wrinkles like),
 (silk spot, has appearance, wave like plaque),
 (defect, has appearance, appearance),
 (inclusion, has appearance, spot),
 (crescent gap, has appearance, half circle),
 (rolled pit, has appearance, periodic bulges or pits),
 (defect, is caused by, cause),
 (punching, is caused by, mechanical failure),
 (crescent gap, is caused by, cutting),
 (crease, is caused by, local yield),
 (waist folding, is caused by, low carbon),
 (oil spot, is caused by, mechanical lubricant),
 (water spot, is caused by, drying),
 (factory, is part of, factory),
 (roller, is part of, machine),
 (machine, is part of, factory),
 (production line, is part of, factory),
 (product, is produced by, factory),
 (steel strip, is produced by, machine)}

In [239]:
found_relations = {
   Relation(
       relation[1], 
       Concept(relation[0]), 
       Concept(relation[2])
       ) for relation in llm_output
}


pipeline = Pipeline(
    spacy_model=nlp,
    pipeline_components=[
        ManualCandidateTermExtraction(
            ct_label_strings_map=ct_concept_label
        ),
        AgglomerativeClusteringConceptExtraction(
            distance_threshold=.4
        )
    ],
    corpus_loader=corpus_loader
)

pipeline.run()

pipeline.kr.relations = found_relations

get_relation_ratio(current_pipeline, expected_relations, comparator_args=comparator_args)



[2024-07-12 00:51:16,158] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2024-07-12 00:51:16,160] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]


(0.058823529411764705, 0.045454545454545456, 0.05128205128205128)